# MCP-Style Hierarchical Search Agent — Video Transcripts (v2: Generic Tools)

This notebook demonstrates an **iterative tool-calling agent** that uses the hierarchical search index
`video-transcripts-hierarchical-v2` to answer questions about solar project meeting recordings.

**Key difference from v1:** Instead of many rigid, hard-coded tools, the agent has **3 generic search
tools** — one per hierarchy level (projects, VoCs, chunks). Each tool exposes the full Azure AI Search
query surface as toggles: query type (simple/full/semantic), vector search, OData filters, field
selection, facets, and target search fields. The LLM decides how to craft each query.

**Pattern:** The LLM reads the data dictionary in its system prompt, decides which level to search,
crafts query parameters (semantic vs keyword, which fields, filters), inspects results, and iterates
until it has enough context — true MCP-style tool use.

In [4]:
# Cell 1 — Setup & Configuration
import os, json, requests
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

load_dotenv(Path.cwd() / ".env")

SEARCH_ENDPOINT   = os.environ["SEARCH_ENDPOINT"].rstrip("/")
INDEX_NAME        = os.environ.get("SEARCH_INDEX", "video-transcripts-hierarchical-v2")
AOAI_ENDPOINT     = os.environ["AOAI_ENDPOINT"]
CHAT_DEPLOYMENT   = os.environ.get("CHAT_DEPLOYMENT", "gpt-4.1")
SEARCH_API        = "2024-07-01"

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

client = AzureOpenAI(
    azure_endpoint=AOAI_ENDPOINT,
    azure_ad_token_provider=token_provider,
    api_version="2024-06-01",
)

print(f"Search endpoint: {SEARCH_ENDPOINT}")
print(f"Index:           {INDEX_NAME}")
print(f"AOAI endpoint:   {AOAI_ENDPOINT}")
print(f"Chat model:      {CHAT_DEPLOYMENT}")

Search endpoint: https://aiss-mcaps-khushi-20260309.search.windows.net
Index:           video-transcripts-hierarchical-v2
AOAI endpoint:   https://sample-proj-resource.services.ai.azure.com
Chat model:      gpt-5.1


## Search Helper
A thin wrapper around the Azure AI Search REST API that our tools will call.

In [5]:
def _search_headers():
    token = credential.get_token("https://search.azure.com/.default").token
    return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

def search_index(body: dict) -> dict:
    """POST a search request to the index and return the JSON response."""
    url = f"{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/search?api-version={SEARCH_API}"
    r = requests.post(url, headers=_search_headers(), json=body, timeout=30)
    r.raise_for_status()
    return r.json()

## Tool Design — 3 Generic Search Tools

| Tool | Purpose |
|---|---|
| `search_projects` | Generic search over project-level docs with full query toggles |
| `search_vocs` | Generic search over VoC/video-level docs with full query toggles |
| `search_chunks` | Generic search over chunk-level docs (transcript passages) with full query toggles |

### Query Toggles (shared by all 3 tools)

| Toggle | Type | Description |
|---|---|---|
| `search_text` | string | Full-text search query. `"*"` = match all |
| `filter` | string | OData `$filter` expression (AND'd with doc_type) |
| `select` | string | Comma-separated fields to return |
| `search_fields` | string | Comma-separated searchable fields to target |
| `query_type` | enum | `"simple"` (default), `"full"` (Lucene), `"semantic"` (reranker) |
| `semantic_config` | string | Override default semantic config |
| `use_vector` | boolean | Enable vector search alongside text search |
| `vector_text` | string | Text for vector query (defaults to `search_text`) |
| `vector_fields` | string | Override default vector field |
| `top` | int | Max results |
| `orderby` | string | Sort expression |
| `facets` | array | Facet expressions for aggregation |
| `include_count` | boolean | Include total count in response |

In [ ]:
# ── Generic search engine ──────────────────────────────────────────────────────

DEFAULT_SEMANTIC = {"project": "project-semantic", "voc": "voc-semantic", "chunk": "chunk-semantic"}
DEFAULT_VECTOR   = {"project": "short_project_summary_vector", "voc": "short_VoC_summary_vector", "chunk": "chunk_text_vector"}# Title fields — fuzzy matching is auto-applied when searching these
TITLE_FIELDS = {"project_title", "VoC_title", "Voc_segment_title"}

import re

def _fuzzify(search_text: str) -> str:
    """Append ~1 to each word for Lucene fuzzy matching. Preserves quoted phrases."""
    # Don't fuzzify wildcards or already-fuzzified terms
    if search_text == "*":
        return search_text
    parts = []
    in_quote = False
    for token in re.split(r'(\s+|")', search_text):
        if token == '"':
            in_quote = not in_quote
            parts.append(token)
        elif in_quote or not token.strip():
            parts.append(token)
        elif token.endswith("~") or token.endswith("~1") or token.endswith("~2"):
            parts.append(token)  # already fuzzy
        elif token in ("+", "-", "|", "AND", "OR", "NOT"):
            parts.append(token)  # boolean operators
        else:
            parts.append(f"{token}~1")
    return "".join(parts)


def _execute_search(
    doc_type: str,
    search_text: str = "*",
    filter: str = None,
    select: str = None,
    search_fields: str = None,
    query_type: str = "simple",
    semantic_config: str = None,
    use_vector: bool = False,
    vector_text: str = None,
    vector_fields: str = None,
    top: int = 10,
    orderby: str = None,
    facets: list = None,
    include_count: bool = True,
    fuzzy: bool = False,
) -> str:
    """Build and execute an Azure AI Search query scoped to a doc_type."""
    # Always scope to the right hierarchy level
    base_filter = f"doc_type eq '{doc_type}'"
    if filter:
        base_filter += f" and ({filter})"

    # Auto-enable fuzzy when searching title fields (unless vector/semantic already handles it)
    effective_query_type = query_type
    effective_search_text = search_text

    if fuzzy and search_text != "*":
        effective_search_text = _fuzzify(search_text)
        if query_type == "simple":
            effective_query_type = "full"  # fuzzy requires Lucene syntax

    body = {
        "search": effective_search_text,
        "filter": base_filter,
        "top": top,
        "count": include_count,
    }

    if select:
        body["select"] = select
    if search_fields:
        body["searchFields"] = search_fields
    if orderby:
        body["orderby"] = orderby

    # Query type
    if effective_query_type == "semantic":
        body["queryType"] = "semantic"
        body["semanticConfiguration"] = semantic_config or DEFAULT_SEMANTIC[doc_type]
    elif effective_query_type == "full":
        body["queryType"] = "full"

    # Vector search
    if use_vector:
        v_fields = vector_fields or DEFAULT_VECTOR[doc_type]
        v_text = vector_text or search_text  # use original text for vector, not fuzzified
        body["vectorQueries"] = [{
            "kind": "text",
            "text": v_text,
            "fields": v_fields,
            "k": top,
        }]

    # Facets
    if facets:
        body["facets"] = facets if isinstance(facets, list) else [facets]

    resp = search_index(body)
    results = resp.get("value", [])

    output = {}
    if include_count:
        output["count"] = resp.get("@odata.count")
    if facets and "@search.facets" in resp:
        output["facets"] = resp["@search.facets"]
    output["results"] = results

    return json.dumps(output, indent=2)


# ── Three level-specific search wrappers ──────────────────────────────────────

def search_projects(**kw) -> str:
    return _execute_search("project", **kw)

def search_vocs(**kw) -> str:
    return _execute_search("voc", **kw)

def search_chunks(**kw) -> str:
    return _execute_search("chunk", **kw)


TOOL_DISPATCH = {
    "search_projects": lambda **kw: search_projects(**kw),
    "search_vocs":     lambda **kw: search_vocs(**kw),
    "search_chunks":   lambda **kw: search_chunks(**kw),
}

print(f"Registered tools: {list(TOOL_DISPATCH.keys())}")

Registered tools: ['search_projects', 'search_vocs', 'search_chunks']


In [ ]:
# ── Shared parameter definitions ──────────────────────────────────────────────

_SEARCH_PARAMS = {
    "search_text": {
        "type": "string",
        "description": "Full-text search query. Use '*' for match-all. Simple syntax supports +required -excluded | OR \"exact phrase\". Full (Lucene) syntax adds field:value, wildcards, fuzzy~, proximity~N, regex /pattern/.",
    },
    "filter": {
        "type": "string",
        "description": "OData $filter expression (AND'd with the auto doc_type filter). Examples: \"project_id eq 'abc123'\", \"voc_category eq 'exploratory'\", \"Video_key_words/any(k: k eq 'IV curve')\". See data dictionary for filterable fields.",
    },
    "select": {
        "type": "string",
        "description": "Comma-separated field names to return. Omit to get all retrievable fields. Use this to keep responses small — request only what you need.",
    },
    "search_fields": {
        "type": "string",
        "description": "Comma-separated searchable fields to target with search_text. Omit to search all searchable fields. Use for precision: e.g. 'VoC_title' to match on title only.",
    },
    "query_type": {
        "type": "string",
        "enum": ["simple", "full", "semantic"],
        "description": "simple: keyword matching (default). full: Lucene syntax (field:, regex, fuzzy). semantic: activates AI semantic reranker for meaning-based relevance.",
    },
    "semantic_config": {
        "type": "string",
        "description": "Override semantic configuration. Only used when query_type='semantic'. Defaults: project-semantic, voc-semantic, chunk-semantic.",
    },
    "use_vector": {
        "type": "boolean",
        "description": "Enable vector (embedding) search alongside text search for hybrid retrieval. Default false.",
    },
    "vector_text": {
        "type": "string",
        "description": "Text to embed for vector search. Defaults to search_text. Set separately when search_text uses filter syntax or '*'.",
    },
    "vector_fields": {
        "type": "string",
        "description": "Vector field to search. Defaults per level: short_project_summary_vector (project), short_VoC_summary_vector (voc), chunk_text_vector (chunk). Chunks also have Voc_segment_summary_vector.",
    },
    "top": {
        "type": "integer",
        "description": "Max results to return.",
    },
    "orderby": {
        "type": "string",
        "description": "Sort expression, e.g. 'chunk_index asc'. Only works on sortable fields.",
    },
    "facets": {
        "type": "array",
        "items": {"type": "string"},
        "description": "Facet expressions for aggregation, e.g. ['voc_category', 'project_title']. Returns value counts.",
    },
    "include_count": {
        "type": "boolean",
        "description": "Include total matching document count in response. Default true.",
    },
    "fuzzy": {
        "type": "boolean",
        "description": "Enable fuzzy matching (edit distance 1) for typo tolerance. Recommended when searching by name or title (project_title, VoC_title) and you suspect typos or get zero results. Automatically converts to Lucene syntax. Default false.",
    },
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_projects",
            "description": (
                "Search project-level documents (doc_type='project'). "
                "Searchable fields: project_title, short_project_summary, project_key_words, project_product_names. "
                "Filterable: id, project_title, num_VoCs, project_key_words, project_product_names. "
                "Vector: short_project_summary_vector. Semantic config: project-semantic. "
                "Key retrievable fields: id, project_title, short_project_summary, long_stakeholder_project_summary, num_VoCs, project_key_words, project_product_names."
            ),
            "parameters": {
                "type": "object",
                "properties": _SEARCH_PARAMS,
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_vocs",
            "description": (
                "Search VoC/video-level documents (doc_type='voc'). "
                "Searchable fields: VoC_title, short_VoC_summary, segment_chronological_summary, short_project_summary_denorm, project_title, Video_key_words, Video_product_names. "
                "Filterable: id, voc_id, project_id, project_title, VoC_title, voc_category, Video_product_names, Video_key_words. "
                "Vector: short_VoC_summary_vector. Semantic config: voc-semantic (title=VoC_title, content=short_VoC_summary+segment_chronological_summary+short_project_summary_denorm+project_title). "
                "Key retrievable fields: id, voc_id, VoC_title, project_title, short_VoC_summary, segment_chronological_summary, long_stakeholder_VoC_summary, voc_category, Video_key_words, Video_product_names."
            ),
            "parameters": {
                "type": "object",
                "properties": _SEARCH_PARAMS,
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_chunks",
            "description": (
                "Search chunk-level documents (doc_type='chunk') — timestamped transcript passages. "
                "Searchable fields: chunk_text, VoC_title, project_title, Voc_segment_title, Voc_segment_summary, short_VoC_summary. "
                "Filterable: id, project_id, voc_id, project_title, VoC_title, Voc_segment_title, chunk_index, chunk_start_time, chunk_end_time, voc_category. "
                "Vectors: chunk_text_vector (default), Voc_segment_summary_vector. Semantic config: chunk-semantic. "
                "Key retrievable fields: id, chunk_index, chunk_text, chunk_start_time, chunk_end_time, VoC_title, project_title, Voc_segment_title, Voc_segment_summary, short_VoC_summary, voc_category."
            ),
            "parameters": {
                "type": "object",
                "properties": _SEARCH_PARAMS,
                "required": [],
            },
        },
    },
]

print(f"{len(TOOLS)} tool schemas defined")

3 tool schemas defined


## Agent Loop

The agent has **3 generic search tools** — one per hierarchy level (projects, VoCs, chunks) —
with full query control. The system prompt provides a data dictionary so the LLM knows what
fields exist, which are filterable/searchable/vectorizable, and what semantic configs are
available. The LLM crafts queries by setting toggles — it can do keyword search, semantic
search, vector search, OData filtering, facet aggregation, or any combination. Discovery
(listing projects/VoCs) is done through the same search tools with `search_text="*"` and
appropriate `select` fields.

In [ ]:
SYSTEM_PROMPT = """You are a research assistant with access to a hierarchical Azure AI Search index
of meeting video transcripts. You have 3 search tools and full control over how to query.
Think of yourself as an MCP client: craft the right query, inspect results, refine, and iterate
until you have enough context to answer.

You do NOT know what projects, videos, or content exist in the index. You must discover
everything through your tools.

═══════════════════════════════════════════════════════════════
TOOLS
═══════════════════════════════════════════════════════════════

1. search_projects(...)  — search project-level docs
2. search_vocs(...)      — search VoC/video-level docs
3. search_chunks(...)    — search chunk-level docs (transcript passages)

All 3 tools share the same query toggles. You control the full search surface:
text queries, OData filters, field selection, query type, vector search, facets, etc.

═══════════════════════════════════════════════════════════════
DATA DICTIONARY
═══════════════════════════════════════════════════════════════

The index has 3 hierarchy levels, all in one flat index differentiated by doc_type:

PROJECT (doc_type='project')
  Searchable:  project_title, short_project_summary, project_key_words, project_product_names
  Filterable:  id, project_title, num_VoCs, project_key_words, project_product_names
  Retrievable: id, project_title, short_project_summary, long_stakeholder_project_summary,
               num_VoCs, project_key_words, project_product_names
  Vector:      short_project_summary_vector (1536d, source: short_project_summary)
  Semantic:    project-semantic (content=short_project_summary, title=project_title,
               keywords=project_key_words+project_product_names)

VOC (doc_type='voc') — one doc per video recording
  Searchable:  VoC_title, short_VoC_summary, segment_chronological_summary,
               short_project_summary_denorm, project_title, Video_key_words, Video_product_names
  Filterable:  id, voc_id, project_id, project_title, VoC_title, voc_category,
               Video_product_names, Video_key_words
  Retrievable: id, voc_id, VoC_title, project_title, short_VoC_summary,
               segment_chronological_summary, long_stakeholder_VoC_summary, voc_category,
               short_project_summary_denorm, Video_product_names, Video_key_words
  Vector:      short_VoC_summary_vector (1536d, source: short_VoC_summary)
  Semantic:    voc-semantic (content=short_VoC_summary+segment_chronological_summary
               +short_project_summary_denorm+project_title, title=VoC_title,
               keywords=Video_key_words+Video_product_names)
  Categories:  voc_category ∈ {exploratory, operational, combinational}

CHUNK (doc_type='chunk') — timestamped transcript passages
  Searchable:  chunk_text, VoC_title, project_title, Voc_segment_title,
               Voc_segment_summary, short_VoC_summary
  Filterable:  id, project_id, voc_id, project_title, VoC_title, Voc_segment_title,
               chunk_index (sortable), chunk_start_time, chunk_end_time, voc_category,
               Voc_segment_start_time, Voc_segment_end_time
  Retrievable: id, chunk_index, chunk_text, chunk_start_time, chunk_end_time,
               VoC_title, project_title, Voc_segment_title, Voc_segment_summary,
               Voc_segment_start_time, Voc_segment_end_time, short_VoC_summary, voc_category
  Vectors:     chunk_text_vector (1536d, source: chunk_text) — DEFAULT
               Voc_segment_summary_vector (1536d, source: Voc_segment_summary)
  Semantic:    chunk-semantic (content=chunk_text+Voc_segment_summary+short_VoC_summary,
               title=VoC_title, keywords=Voc_segment_title+project_title)

LINKING FIELDS (present on VoC + Chunk docs):
  project_id → project doc's id    |  voc_id → VoC doc's id
  parent_id  → immediate parent's id

IDs are opaque SHA-256 hashes (48 chars), NOT human-readable titles.

IMPORTANT — TITLE FORMATS:
  project_title and VoC_title are full descriptive names, NOT short codes.
  You do NOT know their exact values. A user saying "project X" does NOT mean
  project_title eq 'X' — titles may be longer or formatted differently.
  NEVER hard-code a title value in an OData eq filter. Always resolve the exact
  value first via a search (see SCOPED SEARCH pattern below).

═══════════════════════════════════════════════════════════════
QUERY TOOLKIT — HOW TO USE THE TOGGLES
═══════════════════════════════════════════════════════════════

Each search tool accepts the same toggles. Pick the combination that fits your goal:

FINDING A VIDEO BY NAME:
  → search_vocs(search_text="<user's term>", search_fields="VoC_title",
      select="id,VoC_title", top=3)
  → If zero results or unsure about spelling, retry with fuzzy=True
     and/or use_vector=True for typo tolerance
  → Then get full summary: search_vocs(filter="id eq '<id_from_above>'",
      select="id,VoC_title,long_stakeholder_VoC_summary")

SUMMARIZING A PROJECT:
  → search_projects(search_text="<user's term>", search_fields="project_title",
      select="id,project_title", top=3)
  → If zero results or unsure about spelling, retry with fuzzy=True
     and/or use_vector=True for typo tolerance
  → Then: search_projects(filter="id eq '<resolved_id>'",
      select="id,project_title,long_stakeholder_project_summary")

TOPIC SEARCH ACROSS ALL CHUNKS:
  → search_chunks(search_text="<natural language question>", query_type="semantic",
      use_vector=True, top=10,
      select="id,chunk_text,VoC_title,project_title,chunk_start_time,chunk_end_time")

SCOPED SEARCH (topic within a specific project):
  Step 1 — resolve the project's exact title, id, and num_VoCs:
  → search_projects(search_text="<user's term>", search_fields="project_title",
      select="id,project_title,num_VoCs", top=3)
  → If zero results, retry with fuzzy=True and/or use_vector=True
  Step 2 — search within that project:
  → search_chunks(search_text="<topic>", filter="project_id eq '<id_from_step1>'",
      query_type="semantic", use_vector=True, top=<adjust based on scope>,
      select="id,chunk_text,VoC_title,chunk_start_time,chunk_end_time,Voc_segment_title")

RETRIEVING SPECIFIC SEGMENTS OF A VIDEO:
  Segments ≠ chunks. Each segment contains multiple chunks. You do NOT know how
  many chunks each segment has, so you cannot use chunk_index to identify segments.
  Step 1 — resolve the VoC and get its segment structure:
  → search_vocs(search_text="<user's term>", search_fields="VoC_title",
      select="id,VoC_title,segment_chronological_summary", top=3)
  → The segment_chronological_summary field lists all segments in chronological
    order with their titles and timestamp ranges. Read it to identify the
    exact Voc_segment_title values for the requested segments (e.g. "first 2
    segments" = the first 2 segment titles listed in the summary).
  Step 2 — fetch chunks for those specific segments:
  → search_chunks(search_text="*",
      filter="voc_id eq '<id>' and Voc_segment_title eq '<segment_title>'",
      orderby="chunk_index asc", top=10,
      select="id,chunk_text,Voc_segment_title,Voc_segment_summary,chunk_start_time,chunk_end_time,chunk_index")
  → For multiple segments, make one call per segment or use:
    filter="voc_id eq '<id>' and (Voc_segment_title eq '<title1>' or Voc_segment_title eq '<title2>')"

LISTING ALL PROJECTS:
  → search_projects(search_text="*", select="id,project_title,num_VoCs",
      include_count=True)

LISTING VoCs IN A PROJECT:
  → search_vocs(search_text="*", filter="project_id eq '<id>'",
      select="id,VoC_title,voc_category", include_count=True)

COLLECTION FIELD FILTERS:
  → filter="Video_key_words/any(k: k eq '<keyword>')"
  → filter="Video_product_names/any(p: search.in(p, '<name1>,<name2>'))"

QUERY TYPES:
  simple  — keyword matching. +required -excluded | OR "exact phrase"
  full    — Lucene syntax: field:value, wildcards*, fuzzy~, proximity~N, regex /pat/
  semantic — AI reranker for meaning-based relevance. Best for natural language questions.
             Combine with use_vector=True for hybrid (vector + semantic rerank).

═══════════════════════════════════════════════════════════════
CRITICAL RULES
═══════════════════════════════════════════════════════════════

1. ALWAYS use `select` to retrieve ONLY the fields you need for the current step.
   Never omit it. This prevents token overload and keeps your context clean.
   You can search on ANY searchable field — `select` only controls what comes back.
   Recommended select patterns:
   - Name/ID lookup:    select="id,project_title"  or  select="id,VoC_title"
   - Summary fetch:     select="id,VoC_title,long_stakeholder_VoC_summary"
   - Chunk evidence:    select="id,chunk_text,VoC_title,project_title,chunk_start_time,chunk_end_time"
   - Keyword discovery: select="id,VoC_title,Video_key_words,Video_product_names"

2. Adjust `top` based on what you know about the data. After resolving a project,
   use its num_VoCs to set appropriate top values:
   - If a project has 6 VoCs, set top=6 when listing its VoCs
   - For chunk searches within a small project, top=8-10 is usually enough
   - For broad cross-project chunk searches, top=10-15
   - Never request more results than you need

3. Use search_fields to target specific fields (e.g. VoC_title for name lookup).

4. long_stakeholder_project_summary and long_stakeholder_VoC_summary are pre-generated
   rich summaries. Retrieve and return them directly for summary requests — do not
   search chunks to manually build a summary when these exist.

5. For cross-project comparisons, run separate scoped searches per project.

6. CITATIONS — cite only the sources you actually use in your answer:
   - For chunk content: cite the VoC_title (video name) and timestamp range
     (chunk_start_time – chunk_end_time)
   - For VoC-level content: cite the VoC_title
   - For project-level content: cite the project_title
   - Do NOT list all fetched documents — only cite what directly supports
     your answer. If a fetched result was not relevant, do not mention it.

7. If you don't know something, say so — do not guess.

═══════════════════════════════════════════════════════════════
EFFICIENCY RULES — MINIMIZE TOOL CALLS
═══════════════════════════════════════════════════════════════

8. MAXIMIZE work per tool call. Each tool call costs tokens and latency.
   Your target is 1-2 tool calls for simple questions, 2-4 for complex ones.
   NEVER exceed 4 tool calls unless a previous call returned zero results and
   you need to try a genuinely different strategy.

9. NEVER split a question into sub-queries. Semantic and vector search already
   handle meaning-matching — you do NOT need separate queries for synonyms,
   related terms, or sub-topics. For example:
   BAD:  3 calls — "commissioning", "challenges", "solar commissioning"
   GOOD: 1 call  — "challenges regarding solar commissioning"
   The semantic reranker and vector similarity will find relevant passages from
   a single well-formed natural language query.

10. Pick ONE hierarchy level to search first — the one most likely to answer
    the question. Do NOT search the same content at multiple levels (chunks AND
    VoCs) in the same iteration unless the first search returned insufficient results.
    - For detailed evidence / transcript quotes → search_chunks
    - For video-level overviews / summaries → search_vocs
    - For project-level summaries → search_projects

    - ALWAYS resolve the exact title first: do ONE lookup call
      (search_projects with search_text="<user's term>",
      search_fields="project_title", select="id,project_title,num_VoCs",
      top=3)
      to get the exact project_title, id, and num_VoCs.
    - Then use the resolved id in a filter: filter="project_id eq '<id>'"
      (project_id is available on VoC and chunk docs).
    - This is a 2-call pattern: 1 lookup + 1 scoped search.
    - If the lookup returns zero results, use fuzzy=True (enables
      edit-distance matching for typos like "thiunderstruck" → "Thunderstruck")
      and/or use_vector=True (embedding-based fallback) and retry.

12. Only iterate if the previous call's results are genuinely insufficient.
    Before making another call, ask: "Do I already have enough to answer?"
    If yes, stop and synthesize the answer."""

MAX_ITERATIONS = 8
MAX_TOTAL_TOKENS = 20_000
def run_agent(user_question: str, verbose: bool = False):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_question},
    ]

    total_tokens = 0
    tool_call_log = []
    document_log = []

    for iteration in range(1, MAX_ITERATIONS + 1):
        resp = client.chat.completions.create(
            model=CHAT_DEPLOYMENT,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            temperature=0,
        )
        choice = resp.choices[0]
        total_tokens += resp.usage.total_tokens if resp.usage else 0

        # If model produced a text answer, we're done
        if choice.finish_reason == "stop" or not choice.message.tool_calls:
            final = choice.message.content or "(no answer)"
            print(f"\n{'='*60}")
            print(f"ANSWER (iterations={iteration}, tokens={total_tokens}):")
            print(f"{'='*60}")
            print(final)
            if verbose:
                print(f"\n--- Full conversation ({len(messages)} messages) ---")
                for m in messages:
                    role = m.get("role", m.get("type", "?"))
                    content = str(m.get("content", ""))[:200]
                    print(f"  [{role}] {content}")
            return {"answer": final, "tool_calls": tool_call_log, "documents": document_log}

        # Execute tool calls
        messages.append(choice.message)
        for tc in choice.message.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments) if tc.function.arguments else {}
            print(f"  [{iteration}] Calling {fn_name}({fn_args})")
            # Log tool call and result
            if fn_name in TOOL_DISPATCH:
                result = TOOL_DISPATCH[fn_name](**fn_args)
            else:
                result = json.dumps({"error": f"Unknown tool: {fn_name}"})

            # Log tool call and result
            tool_call_log.append({"iteration": iteration, "tool": fn_name, "args": fn_args})
            try:
                parsed = json.loads(result)
                document_log.append({"iteration": iteration, "tool": fn_name, "result": parsed})
            except json.JSONDecodeError:
                document_log.append({"iteration": iteration, "tool": fn_name, "result": result})

            # Note: no truncation — select fields carefully to control result size

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            })

        if total_tokens > MAX_TOTAL_TOKENS:
            print(f"Token budget exceeded ({total_tokens}). Stopping.")
            break

    return {"answer": "Agent did not converge to an answer within the iteration limit.", "tool_calls": tool_call_log, "documents": document_log}

## Example Queries

Try different questions to see how the agent navigates the hierarchy.

In [ ]:
# Exploratory: what's in the index?
result = run_agent("What projects and video recordings are available? Give me an overview.")

  [1] Calling search_projects({'search_text': '*', 'select': 'id,project_title,num_VoCs', 'top': 50, 'include_count': True})
  [2] Calling search_vocs({'search_text': '*', 'filter': "project_id eq '71cfe30febe40daadc2c21ab242cf2ea014f83a0d5a7004e'", 'select': 'id,VoC_title,voc_category', 'top': 6, 'include_count': True})
  [2] Calling search_vocs({'search_text': '*', 'filter': "project_id eq '7734bf224d51369f0b9b9d16b10a7304cfe512eeb1a3f63f'", 'select': 'id,VoC_title,voc_category', 'top': 6, 'include_count': True})

ANSWER (iterations=3, tokens=15540):
Here’s an overview of what’s available in the index right now.

## Projects

1. **ATUM (Ra III 1500V Solar)**  
   - **Number of video recordings (VoCs):** 6  

2. **Thunderstruck (2000V Solar)**  
   - **Number of video recordings (VoCs):** 6  

---

## Video recordings by project

### 1. ATUM (Ra III 1500V Solar) — 6 VoCs

- **Jose Luis and Ankur Bakshi-20230721_090820-Meeting Recording**  
  - Category: **combinational** (mix of disco

In [10]:
# Tool calls made & queries passed
import pandas as pd
if result["tool_calls"]:
    df = pd.DataFrame(result["tool_calls"])
    df["args"] = df["args"].apply(lambda x: json.dumps(x) if x else "{}")
    display(df.style.set_caption("Tool Calls — Overview Query"))
else:
    print("No tool calls were made.")

,iteration,tool,args
0,1,search_projects,"{""search_text"": ""*"", ""select"": ""id,project_title,num_VoCs"", ""top"": 50, ""include_count"": true}"
1,2,search_vocs,"{""search_text"": ""*"", ""filter"": ""project_id eq '71cfe30febe40daadc2c21ab242cf2ea014f83a0d5a7004e'"", ""select"": ""id,VoC_title,voc_category"", ""top"": 6, ""include_count"": true}"
2,2,search_vocs,"{""search_text"": ""*"", ""filter"": ""project_id eq '7734bf224d51369f0b9b9d16b10a7304cfe512eeb1a3f63f'"", ""select"": ""id,VoC_title,voc_category"", ""top"": 6, ""include_count"": true}"


In [12]:
# Documents fetched from tool calls
for i, doc in enumerate(result["documents"]):
    print(f"\n{'='*60}")
    print(f"[Iteration {doc['iteration']}] {doc['tool']} — Response:")
    print(f"{'='*60}")
    print(json.dumps(doc["result"], indent=2))


[Iteration 1] search_projects — Response:
{
  "count": 2,
  "results": [
    {
      "@search.score": 1.0,
      "id": "71cfe30febe40daadc2c21ab242cf2ea014f83a0d5a7004e",
      "project_title": "ATUM (Ra III 1500V Solar)",
      "num_VoCs": 6
    },
    {
      "@search.score": 1.0,
      "id": "7734bf224d51369f0b9b9d16b10a7304cfe512eeb1a3f63f",
      "project_title": "Thunderstruck (2000V Solar)",
      "num_VoCs": 6
    }
  ]
}

[Iteration 2] search_vocs — Response:
{
  "count": 6,
  "results": [
    {
      "@search.score": 1.0,
      "id": "f3374e28b319c285850cecee9cce760e1585364c86523b73",
      "VoC_title": "Jose Luis and Ankur Bakshi-20230721_090820-Meeting Recording",
      "voc_category": "combinational"
    },
    {
      "@search.score": 1.0,
      "id": "db32d10d67fda77b66fb8a24b14ae3be69278438b4b6b861",
      "VoC_title": "Hanna and Fluke Solar Team-20230802_122308-Meeting Recording",
      "voc_category": "exploratory"
    },
    {
      "@search.score": 1.0,
      "id":

In [ ]:
# Single-project question
result = run_agent("What are the main challenges discussed in the ATUM project regarding solar commissioning?")

  [1] Calling search_projects({'search_text': 'ATUM', 'search_fields': 'project_title', 'select': 'id,project_title,num_VoCs', 'query_type': 'simple', 'top': 5})
  [2] Calling search_chunks({'search_text': 'challenges regarding solar commissioning, issues during commissioning, commissioning problems, lessons learned', 'filter': "project_id eq '71cfe30febe40daadc2c21ab242cf2ea014f83a0d5a7004e'", 'select': 'id,chunk_text,VoC_title,project_title,chunk_start_time,chunk_end_time', 'query_type': 'semantic', 'use_vector': True, 'top': 12})

ANSWER (iterations=3, tokens=21778):
From the ATUM (Ra III 1500V Solar) discussions, the main commissioning‑related challenges fall into a few clear buckets:

1. **Grounding and bonding issues (very frequent)**
   - Subarrays are supposed to be fully bonded, but loose or missing grounds are “missed pretty often” by contractors and only discovered later by service techs. This is especially critical with module‑level electronics (MLPE/MLRSD), where poor grou

In [14]:
# Tool calls made & queries passed
if result["tool_calls"]:
    df = pd.DataFrame(result["tool_calls"])
    df["args"] = df["args"].apply(lambda x: json.dumps(x) if x else "{}")
    display(df.style.set_caption("Tool Calls — ATUM Commissioning Query"))
else:
    print("No tool calls were made.")

,iteration,tool,args
0,1,search_projects,"{""search_text"": ""ATUM"", ""search_fields"": ""project_title"", ""select"": ""id,project_title,num_VoCs"", ""query_type"": ""simple"", ""top"": 5}"
1,2,search_chunks,"{""search_text"": ""challenges regarding solar commissioning, issues during commissioning, commissioning problems, lessons learned"", ""filter"": ""project_id eq '71cfe30febe40daadc2c21ab242cf2ea014f83a0d5a7004e'"", ""select"": ""id,chunk_text,VoC_title,project_title,chunk_start_time,chunk_end_time"", ""query_type"": ""semantic"", ""use_vector"": true, ""top"": 12}"


In [15]:
# Documents fetched from tool calls
for i, doc in enumerate(result["documents"]):
    print(f"\n{'='*60}")
    print(f"[Iteration {doc['iteration']}] {doc['tool']} — Response:")
    print(f"{'='*60}")
    print(json.dumps(doc["result"], indent=2))


[Iteration 1] search_projects — Response:
{
  "count": 1,
  "results": [
    {
      "@search.score": 0.8390806,
      "id": "71cfe30febe40daadc2c21ab242cf2ea014f83a0d5a7004e",
      "project_title": "ATUM (Ra III 1500V Solar)",
      "num_VoCs": 6
    }
  ]
}

[Iteration 2] search_chunks — Response:
{
  "count": 139,
  "results": [
    {
      "@search.score": 0.014084506779909134,
      "@search.rerankerScore": 2.550845146179199,
      "id": "2da407de3d59bccd4b7d64f1bc79cc8aed12012f9c25a659",
      "project_title": "ATUM (Ra III 1500V Solar)",
      "VoC_title": "Skylar Carreon and Fluke Solar Team-20230804_130031-Meeting Recording",
      "chunk_text": "So we'll go from subarray to a subarray or array to array. You know, we'll have like a 500 foot lead and we kind of run it like that. And then of course we Meg, of course, you know, and that's kind of, that's not really the same thing. But you know, we're going to the frame of the of the of the mod or whatever we got to do the racki

In [16]:
# Cross-project question
result = run_agent("Compare how IV curve tracing is discussed across the ATUM and Thunderstruck projects. What tools and workflows are mentioned?")

  [1] Calling search_projects({'search_text': 'ATUM', 'search_fields': 'project_title', 'select': 'id,project_title,num_VoCs', 'top': 3})
  [1] Calling search_projects({'search_text': 'Thunderstruck', 'search_fields': 'project_title', 'select': 'id,project_title,num_VoCs', 'top': 3})
  [2] Calling search_chunks({'search_text': 'IV curve tracing tools workflows commissioning testing', 'filter': "project_id eq '71cfe30febe40daadc2c21ab242cf2ea014f83a0d5a7004e'", 'query_type': 'semantic', 'use_vector': True, 'top': 10, 'select': 'id,chunk_text,VoC_title,project_title,chunk_start_time,chunk_end_time'})
  [2] Calling search_chunks({'search_text': 'IV curve tracing tools workflows commissioning testing', 'filter': "project_id eq '7734bf224d51369f0b9b9d16b10a7304cfe512eeb1a3f63f'", 'query_type': 'semantic', 'use_vector': True, 'top': 10, 'select': 'id,chunk_text,VoC_title,project_title,chunk_start_time,chunk_end_time'})

ANSWER (iterations=3, tokens=27587):
Here’s a side‑by‑side comparison of

In [17]:
# Tool calls made & queries passed
if result["tool_calls"]:
    df = pd.DataFrame(result["tool_calls"])
    df["args"] = df["args"].apply(lambda x: json.dumps(x) if x else "{}")
    display(df.style.set_caption("Tool Calls — IV Curve Cross-Project Query"))
else:
    print("No tool calls were made.")

,iteration,tool,args
0,1,search_projects,"{""search_text"": ""ATUM"", ""search_fields"": ""project_title"", ""select"": ""id,project_title,num_VoCs"", ""top"": 3}"
1,1,search_projects,"{""search_text"": ""Thunderstruck"", ""search_fields"": ""project_title"", ""select"": ""id,project_title,num_VoCs"", ""top"": 3}"
2,2,search_chunks,"{""search_text"": ""IV curve tracing tools workflows commissioning testing"", ""filter"": ""project_id eq '71cfe30febe40daadc2c21ab242cf2ea014f83a0d5a7004e'"", ""query_type"": ""semantic"", ""use_vector"": true, ""top"": 10, ""select"": ""id,chunk_text,VoC_title,project_title,chunk_start_time,chunk_end_time""}"
3,2,search_chunks,"{""search_text"": ""IV curve tracing tools workflows commissioning testing"", ""filter"": ""project_id eq '7734bf224d51369f0b9b9d16b10a7304cfe512eeb1a3f63f'"", ""query_type"": ""semantic"", ""use_vector"": true, ""top"": 10, ""select"": ""id,chunk_text,VoC_title,project_title,chunk_start_time,chunk_end_time""}"


In [18]:
# Documents fetched from tool calls
for i, doc in enumerate(result["documents"]):
    print(f"\n{'='*60}")
    print(f"[Iteration {doc['iteration']}] {doc['tool']} — Response:")
    print(f"{'='*60}")
    print(json.dumps(doc["result"], indent=2))


[Iteration 1] search_projects — Response:
{
  "count": 1,
  "results": [
    {
      "@search.score": 0.8390806,
      "id": "71cfe30febe40daadc2c21ab242cf2ea014f83a0d5a7004e",
      "project_title": "ATUM (Ra III 1500V Solar)",
      "num_VoCs": 6
    }
  ]
}

[Iteration 1] search_projects — Response:
{
  "count": 1,
  "results": [
    {
      "@search.score": 0.33611667,
      "id": "7734bf224d51369f0b9b9d16b10a7304cfe512eeb1a3f63f",
      "project_title": "Thunderstruck (2000V Solar)",
      "num_VoCs": 6
    }
  ]
}

[Iteration 2] search_chunks — Response:
{
  "count": 139,
  "results": [
    {
      "@search.score": 0.010309278033673763,
      "@search.rerankerScore": 2.998893976211548,
      "id": "324a7db4d0e620bcb616d888c20ba9de6c31a111bc4994a9",
      "project_title": "ATUM (Ra III 1500V Solar)",
      "VoC_title": "Ronald Hamski and Fluke Solar Team-20230726_140227-Meeting Recording",
      "chunk_text": "And again, I reference a national standard. I don't make it up as I go

In [19]:
result = run_agent("Give me summary of PLCMEL~2 video")

  [1] Calling search_vocs({'search_text': 'PLCMEL~2', 'search_fields': 'VoC_title', 'select': 'id,VoC_title,long_stakeholder_VoC_summary', 'query_type': 'semantic', 'top': 3})

ANSWER (iterations=2, tokens=12508):
Here’s the summary of the **PLCMEL~2** video:

- **Who’s on the call / customer profile**  
  Global test & measurement company teams: product managers, commercial teams (Americas, EMEA, APAC/China), and service operations. Their end customers are solar professionals, HVAC contractors, utilities, and industrial users, from small shops with a handful of DMMs/clamps to large fleets with 100+ units. Key regions: US (biggest DMM market), China (unique distribution/compliance), EMEA (master electricians, utilities), APAC (very price-sensitive).

- **Current workflows & tools**  
  - Customers buy DMMs and clamp meters (notably **87.5 DMM** and **393 FC** for 2000V solar) mainly via distributors (Digikey, Grainger) and e‑commerce.  
  - Product registration happens via the **Unifie

In [20]:
result = run_agent("Summarize Project Thunderstruck")

  [1] Calling search_projects({'search_text': 'Thunderstruck', 'search_fields': 'project_title', 'select': 'id,project_title,long_stakeholder_project_summary', 'query_type': 'semantic', 'top': 3})

ANSWER (iterations=2, tokens=9954):
Here’s a concise summary of **Project “Thunderstruck (2000V Solar)”**:

Project Thunderstruck is a Voice-of-Customer research initiative focused on **solar test & measurement workflows** as the industry moves to **2000V+ systems and bifacial modules**. It gathers input from utility-scale solar operators, field technicians, product managers, and commercial teams to understand:

- **Measurement tools & hardware needs**
  - Strong demand for **better bifacial support**, especially **dual/backside irradiance sensors** compatible with existing IV curve tracers.
  - Need for **more ergonomic, lighter, and portable IV curve tracers**, even if that means slightly slower operation.
  - Interest in **small-jaw, affordable, high-precision DC clamp meters** suitable f

In [21]:
# Specific topic search
result = run_agent("What do the recordings say about bifacial solar panels and the challenges of testing them?")

  [1] Calling search_chunks({'search_text': 'bifacial solar panels challenges of testing, measurement, or field validation', 'query_type': 'semantic', 'use_vector': True, 'top': 10, 'select': 'id,chunk_text,VoC_title,project_title,chunk_start_time,chunk_end_time'})

ANSWER (iterations=2, tokens=15813):
Across the recordings, people describe several specific challenges with testing bifacial solar panels, mostly around IV‑curve tracing and irradiance/temperature measurement. Key points:

1. **Need to cover the backside to get usable data**  
   - Current IV‑curve tracer irradiance sensors are designed for monofacial modules, so they only measure front‑side irradiance. For bifacial, the backside contribution is missing.  
   - To “standardize” tests, one operator said they have to physically cover the back of each bifacial module so it behaves like a monofacial panel and the irradiance reading makes sense. This is done to get a consistent baseline for performance/degradation analysis.  
 

In [22]:
# Tool calls made & queries passed
if result["tool_calls"]:
    df = pd.DataFrame(result["tool_calls"])
    df["args"] = df["args"].apply(lambda x: json.dumps(x) if x else "{}")
    display(df.style.set_caption("Tool Calls — Bifacial Panels Query"))
else:
    print("No tool calls were made.")

,iteration,tool,args
0,1,search_chunks,"{""search_text"": ""bifacial solar panels challenges of testing, measurement, or field validation"", ""query_type"": ""semantic"", ""use_vector"": true, ""top"": 10, ""select"": ""id,chunk_text,VoC_title,project_title,chunk_start_time,chunk_end_time""}"


In [24]:
# Documents fetched from tool calls
for i, doc in enumerate(result["documents"]):
    print(f"\n{'='*60}")
    print(f"[Iteration {doc['iteration']}] {doc['tool']} — Response:")
    print(f"{'='*60}")
    print(json.dumps(doc["result"], indent=2))


[Iteration 1] search_chunks — Response:
{
  "count": 395,
  "results": [
    {
      "@search.score": 0.03306011110544205,
      "@search.rerankerScore": 2.6238656044006348,
      "id": "07e8ef0df436a5859acfe54b1f9513cb50a75b26c3773e8c",
      "project_title": "Thunderstruck (2000V Solar)",
      "VoC_title": "Enel _ Fluke Solar Tools_Workflow Discussion-20250424_110116-Meeting Recording",
      "chunk_text": "Absolutely. OK. We've actually got one of the units that we just sent into the lab to fix because it wouldn't charge anymore. So if you feel like, you know, beta testing that little sensor on that when you ship back, I wouldn't be opposed. And Speaker 8, if you had that, so you mentioned on a good day you could do about 20 modules and you mentioned the time to cover up the backside. If you didn't have to do that, what does that number change to? Does it go to like 30 or? Yeah. If if we didn't have to cover it up. If it was just disconnect, hook up, read it, move to the next one,

In [ ]:
# Product-focused question
result = run_agent("Which Fluke and Solmetric products are mentioned across all videos - meaning are there any products that are mentioned in every single VoC? Try to search across all vWhat feedback do customers give about them?")

  [1] Calling search_vocs({'search_text': '*', 'select': 'id,VoC_title,Video_product_names', 'top': 1000, 'include_count': True})
  [2] Calling search_chunks({'search_text': 'feedback about Fluke and Solmetric products, likes, dislikes, pros, cons, issues, requests', 'query_type': 'semantic', 'use_vector': True, 'top': 15, 'select': 'id,chunk_text,VoC_title,project_title,chunk_start_time,chunk_end_time'})

ANSWER (iterations=3, tokens=27785):
1) Products mentioned in every VoC

There are 12 VoCs in total. Looking at the `Video_product_names` across all of them, there is **no single Fluke or Solmetric product that appears in every VoC**.

A few patterns:

- **Fluke 393 / 393 FC / 393 clamp meter / Talon (393 FC)**  
  - Appears in multiple VoCs (PLCMEL~2, PLCMEL~1, Ronald Hamski & Fluke Solar Team, Skylar Carreon, Enel workflow, Ronald Hamski Fluke Innovation), but **not all 12**.
- **Fluke SMFT / SMFT 1000 / Fluke SMFT 1000**  
  - Appears in several solar VoCs (Raymond SMFT & IV Curve

In [27]:
# Tool calls made & queries passed
if result["tool_calls"]:
    df = pd.DataFrame(result["tool_calls"])
    df["args"] = df["args"].apply(lambda x: json.dumps(x) if x else "{}")
    display(df.style.set_caption("Tool Calls — Fluke/Solmetric Products Query"))
else:
    print("No tool calls were made.")

,iteration,tool,args
0,1,search_vocs,"{""search_text"": ""*"", ""select"": ""id,VoC_title,Video_product_names"", ""top"": 1000, ""include_count"": true}"
1,2,search_chunks,"{""search_text"": ""feedback about Fluke and Solmetric products, likes, dislikes, pros, cons, issues, requests"", ""query_type"": ""semantic"", ""use_vector"": true, ""top"": 15, ""select"": ""id,chunk_text,VoC_title,project_title,chunk_start_time,chunk_end_time""}"


In [28]:
# Documents fetched from tool calls
for i, doc in enumerate(result["documents"]):
    print(f"\n{'='*60}")
    print(f"[Iteration {doc['iteration']}] {doc['tool']} — Response:")
    print(f"{'='*60}")
    print(json.dumps(doc["result"], indent=2))


[Iteration 1] search_vocs — Response:
{
  "count": 12,
  "results": [
    {
      "@search.score": 1.0,
      "id": "f3374e28b319c285850cecee9cce760e1585364c86523b73",
      "VoC_title": "Jose Luis and Ankur Bakshi-20230721_090820-Meeting Recording",
      "Video_product_names": [
        "Fluke 393 FC",
        "Fluke TI-25 EIR",
        "Fluke 87",
        "Fluke Connect"
      ]
    },
    {
      "@search.score": 1.0,
      "id": "1a32c2692f38ced6f1049232a41dd2afba7c341769af8027",
      "VoC_title": "Enel _ Fluke Solar Tools_Workflow Discussion-20250424_110116-Meeting Recording",
      "Video_product_names": [
        "Fluke IV curve tracer",
        "Solmetric IV curve tracer",
        "PVA 1500",
        "PVA 1000",
        "Solmetric software",
        "Mars 300",
        "393 FC"
      ]
    },
    {
      "@search.score": 1.0,
      "id": "4645ab49e438d675bc7e31bf7fdd48a444dfac0d583c3a65",
      "VoC_title": "Raymond_Fluke SMFT&IV Curve Discussion-20250416_140039-Meeting Reco

In [30]:
# Your own question
result = run_agent("What are the ground fault detection methods discussed across the recordings?")

  [1] Calling search_chunks({'search_text': 'ground fault detection methods, how to detect ground faults, techniques for ground fault detection', 'query_type': 'semantic', 'use_vector': True, 'top': 12, 'select': 'id,chunk_text,VoC_title,project_title,chunk_start_time,chunk_end_time'})

ANSWER (iterations=2, tokens=17140):
Across the recordings, ground faults are mainly detected and localized using a mix of basic electrical measurements and a few specialized tools/techniques:

1. **DC voltage measurements to ground (primary method in PV fields)**  
   - Technicians chase ground faults in PV arrays by systematically measuring DC voltage from each conductor to ground and comparing:  
     - Positive-to-ground vs. negative-to-ground readings  
     - Looking for abnormal or asymmetric voltages that indicate which polarity is faulted and roughly where in the field the fault lies.  
   - This is done with standard DMMs rated for 1000–1500 V (and in future 2000 V) and is described as their “

In [31]:
# Tool calls made & queries passed
if result["tool_calls"]:
    df = pd.DataFrame(result["tool_calls"])
    df["args"] = df["args"].apply(lambda x: json.dumps(x) if x else "{}")
    display(df.style.set_caption("Tool Calls — Ground Fault Detection Query"))
else:
    print("No tool calls were made.")

,iteration,tool,args
0,1,search_chunks,"{""search_text"": ""ground fault detection methods, how to detect ground faults, techniques for ground fault detection"", ""query_type"": ""semantic"", ""use_vector"": true, ""top"": 12, ""select"": ""id,chunk_text,VoC_title,project_title,chunk_start_time,chunk_end_time""}"


In [32]:
# Documents fetched from tool calls
for i, doc in enumerate(result["documents"]):
    print(f"\n{'='*60}")
    print(f"[Iteration {doc['iteration']}] {doc['tool']} — Response:")
    print(f"{'='*60}")
    print(json.dumps(doc["result"], indent=2))


[Iteration 1] search_chunks — Response:
{
  "count": 395,
  "results": [
    {
      "@search.score": 0.03102453239262104,
      "@search.rerankerScore": 2.1919116973876953,
      "id": "b00d5003733a5b8de7a7299f97d8947c281629b755a4d4ea",
      "project_title": "Thunderstruck (2000V Solar)",
      "VoC_title": "Enel _ Fluke Solar Tools_Workflow Discussion-20250424_110116-Meeting Recording",
      "chunk_text": "And I the voltages are going to have to go up and of course currents will go down and OK, instead of 10 gauge wire we can go with 12 or 14 and save a little bit of money in the field wiring side of things, so. OK. Do you have any? Oh, sorry, I was just gonna. In terms of tools required to operate at that 2000 and 2500 Volt range, what would you prioritize I suppose? Really, we don't do a whole lot. The the biggest thing is just voltage readings. When we get there, we have to do a, you know, live dead, live check, make sure everything is good. Obviously, you can't shut off a sola